In [1]:
import fastf1
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ── Load cached session (instant this time) ───────────────
fastf1.Cache.enable_cache('../data/cache')
session = fastf1.get_session(2024, 'Bahrain', 'R')
session.load()

print("✅ Session loaded from cache")
print(f"🏎️  Race: {session.event['EventName']} {session.event.year}")

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']


✅ Session loaded from cache
🏎️  Race: Bahrain Grand Prix 2024


In [2]:
# ── Get clean lap data ────────────────────────────────────
laps = session.laps.copy()
laps['LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()

# ── Filter clean laps only ────────────────────────────────
clean_laps = laps[
    (laps['IsAccurate'] == True) &
    (laps['LapTimeSeconds'] > 85) &
    (laps['LapTimeSeconds'] < 120)
].copy()

# ── Build baseline per driver ─────────────────────────────
baseline = clean_laps.groupby('Driver')['LapTimeSeconds'].agg([
    'mean', 'std', 'min', 'max', 'count'
]).round(3)

baseline.columns = ['AvgLapTime', 'StdDev', 'BestLap', 'WorstLap', 'CleanLaps']
baseline = baseline.sort_values('AvgLapTime')

print("✅ Driver baselines calculated")
print(f"📊 Drivers with clean data: {len(baseline)}")
print()
print(baseline)

✅ Driver baselines calculated
📊 Drivers with clean data: 20

        AvgLapTime  StdDev  BestLap  WorstLap  CleanLaps
Driver                                                  
VER         95.654   1.067   92.608    97.229         52
PER         96.040   1.040   94.364    98.044         52
SAI         96.103   1.064   94.507    98.260         52
LEC         96.368   1.123   94.090    98.957         52
NOR         96.464   1.006   94.476    98.172         52
RUS         96.474   0.886   95.065    98.116         52
HAM         96.495   1.134   94.722    98.645         52
PIA         96.538   1.040   94.774    98.397         52
HUL         96.855   1.067   94.834   100.083         50
ALO         96.940   1.406   94.199    99.325         52
STR         97.124   0.926   95.632    98.925         52
GAS         97.305   1.356   94.805    99.452         49
ZHO         97.407   0.875   95.458    99.646         51
RIC         97.429   1.260   95.163    99.529         51
MAG         97.465   1.064 

In [3]:
# ── Flag anomalous laps per driver ────────────────────────
anomaly_laps = []

for driver in clean_laps['Driver'].unique():
    driver_laps = clean_laps[clean_laps['Driver'] == driver].copy()
    
    avg = baseline.loc[driver, 'AvgLapTime']
    std = baseline.loc[driver, 'StdDev']
    
    # Anomaly threshold: more than 2 standard deviations from mean
    driver_laps['IsAnomaly'] = (
        (driver_laps['LapTimeSeconds'] > avg + 2 * std) |
        (driver_laps['LapTimeSeconds'] < avg - 2 * std)
    )
    
    driver_laps['DeviationSeconds'] = driver_laps['LapTimeSeconds'] - avg
    driver_laps['DeviationSigma'] = driver_laps['DeviationSeconds'] / std
    
    anomaly_laps.append(driver_laps)

all_laps = pd.concat(anomaly_laps, ignore_index=True)
anomalies = all_laps[all_laps['IsAnomaly'] == True].copy()

print(f"✅ Anomaly detection complete")
print(f"🚨 Total anomalous laps detected: {len(anomalies)}")
print(f"📊 Anomalies per driver:")
print(anomalies.groupby('Driver')[['LapNumber', 'LapTimeSeconds', 'DeviationSigma']].count()['LapNumber'].sort_values(ascending=False))
print()
print("🔴 Most extreme anomalies:")
anomalies[['Driver', 'LapNumber', 'LapTimeSeconds', 'DeviationSeconds', 'DeviationSigma', 'Compound']]\
    .sort_values('DeviationSigma', ascending=False).head(10)

✅ Anomaly detection complete
🚨 Total anomalous laps detected: 13
📊 Anomalies per driver:
Driver
SAR    3
ZHO    3
LEC    2
BOT    1
HUL    1
MAG    1
SAI    1
VER    1
Name: LapNumber, dtype: int64

🔴 Most extreme anomalies:


,Driver,LapNumber,LapTimeSeconds,DeviationSeconds,DeviationSigma,Compound
808,HUL,38.0,100.083,3.228,3.025305,HARD
526,ZHO,8.0,99.646,2.239,2.558857,SOFT
576,MAG,7.0,100.058,2.593,2.437030,SOFT
928,BOT,5.0,100.087,2.368,2.431211,SOFT
525,ZHO,7.0,99.517,2.110,2.411429,SOFT
986,SAR,14.0,101.470,3.905,2.320261,HARD
161,LEC,7.0,98.957,2.589,2.305432,SOFT
988,SAR,16.0,101.370,3.805,2.260844,HARD
985,SAR,13.0,100.989,3.424,2.034462,HARD
109,SAI,7.0,98.260,2.157,2.027256,SOFT


In [4]:
# ── Build feature set for Isolation Forest ────────────────
features_for_anomaly = all_laps[['LapTimeSeconds', 'TyreLife', 'LapNumber']].copy()
features_for_anomaly = features_for_anomaly.dropna()

# ── Scale features ────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features_for_anomaly)

# ── Train Isolation Forest ────────────────────────────────
iso_forest = IsolationForest(
    contamination=0.05,    # Expect ~5% of laps to be anomalous
    random_state=42,
    n_estimators=200
)

predictions = iso_forest.fit_predict(X_scaled)
scores = iso_forest.score_samples(X_scaled)

# ── Attach back to dataframe ──────────────────────────────
features_for_anomaly['IsolationAnomaly'] = (predictions == -1)
features_for_anomaly['AnomalyScore'] = scores

# Merge back with original data
all_laps_with_iso = all_laps.dropna(subset=['LapTimeSeconds', 'TyreLife', 'LapNumber']).copy()
all_laps_with_iso['IsolationAnomaly'] = (predictions == -1)
all_laps_with_iso['AnomalyScore'] = scores

# ── Combine both detection methods ────────────────────────
all_laps_with_iso['HighConfidenceAnomaly'] = (
    all_laps_with_iso['IsAnomaly'] & all_laps_with_iso['IsolationAnomaly']
)

iso_anomalies = all_laps_with_iso[all_laps_with_iso['IsolationAnomaly']].copy()
both_flagged = all_laps_with_iso[all_laps_with_iso['HighConfidenceAnomaly']].copy()

print(f"✅ Isolation Forest trained on {len(X_scaled)} laps")
print(f"🔍 Statistical method flagged: {all_laps_with_iso['IsAnomaly'].sum()} laps")
print(f"🤖 Isolation Forest flagged: {len(iso_anomalies)} laps")
print(f"🚨 BOTH methods agree (high confidence): {len(both_flagged)} laps")
print()
print("🔴 High-confidence anomalies (caught by both methods):")
both_flagged[['Driver', 'LapNumber', 'LapTimeSeconds', 'TyreLife', 'Compound', 'DeviationSigma', 'AnomalyScore']]\
    .sort_values('AnomalyScore').head(10)

✅ Isolation Forest trained on 1024 laps
🔍 Statistical method flagged: 13 laps
🤖 Isolation Forest flagged: 52 laps
🚨 BOTH methods agree (high confidence): 8 laps

🔴 High-confidence anomalies (caught by both methods):


,Driver,LapNumber,LapTimeSeconds,TyreLife,Compound,DeviationSigma,AnomalyScore
986,SAR,14.0,101.470,4.0,HARD,2.320261,-0.645177
985,SAR,13.0,100.989,3.0,HARD,2.034462,-0.640447
808,HUL,38.0,100.083,18.0,HARD,3.025305,-0.633859
988,SAR,16.0,101.370,6.0,HARD,2.260844,-0.631003
928,BOT,5.0,100.087,5.0,SOFT,2.431211,-0.606892
33,VER,39.0,92.608,2.0,SOFT,-2.854733,-0.604970
576,MAG,7.0,100.058,7.0,SOFT,2.437030,-0.589174
186,LEC,36.0,94.090,2.0,HARD,-2.028495,-0.580129


In [5]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Sort drivers by avg lap time ──────────────────────────
driver_order = baseline.index.tolist()

fig = go.Figure()

# Color map for compounds
compound_colors = {'SOFT': '#FF3333', 'MEDIUM': '#FFFF00', 'HARD': '#CCCCCC'}

# Plot all clean laps as scatter
for driver in driver_order:
    d_laps = all_laps_with_iso[all_laps_with_iso['Driver'] == driver]
    
    # Normal laps - small grey dots
    normal = d_laps[~d_laps['IsolationAnomaly']]
    fig.add_trace(go.Scatter(
        x=normal['LapNumber'],
        y=[driver] * len(normal),
        mode='markers',
        marker=dict(
            size=6,
            color=[compound_colors.get(c, '#888') for c in normal['Compound']],
            opacity=0.4
        ),
        name=driver,
        showlegend=False,
        hovertemplate=f'{driver}<br>Lap %{{x}}<br>Time: %{{customdata:.3f}}s<extra></extra>',
        customdata=normal['LapTimeSeconds']
    ))
    
    # Anomalous laps - big circles
    anom = d_laps[d_laps['IsolationAnomaly']]
    if len(anom) > 0:
        fig.add_trace(go.Scatter(
            x=anom['LapNumber'],
            y=[driver] * len(anom),
            mode='markers',
            marker=dict(
                size=14,
                color='red',
                symbol='circle-open',
                line=dict(width=3, color='red')
            ),
            name='Anomaly',
            showlegend=False,
            hovertemplate=f'<b>🚨 ANOMALY</b><br>{driver}<br>Lap %{{x}}<br>Time: %{{customdata:.3f}}s<extra></extra>',
            customdata=anom['LapTimeSeconds']
        ))

fig.update_layout(
    title='🚨 Incident Detection — 2024 Bahrain GP<br><sub>Grey dots = Normal | Red circles = Detected Anomalies | Color = Tyre Compound</sub>',
    xaxis_title='Lap Number',
    yaxis_title='Driver',
    template='plotly_dark',
    height=700,
    yaxis=dict(categoryorder='array', categoryarray=driver_order[::-1])
)

fig.show()
print("✅ Incident map rendered")
print(f"📍 {len(iso_anomalies)} incidents detected across {iso_anomalies['Driver'].nunique()} drivers")

✅ Incident map rendered
📍 52 incidents detected across 19 drivers


In [7]:
# ── Add reasoning to each anomaly ─────────────────────────
def explain_anomaly(row):
    reasons = []
    
    # Race start anomaly
    if row['LapNumber'] <= 2:
        reasons.append("🟡 Race start lap (formation effects)")
    
    # Race end anomaly
    if row['LapNumber'] >= 55:
        reasons.append("🟠 End of race (tire cliff)")
    
    # Pit stop in/out lap detection
    if row['TyreLife'] <= 2:
        reasons.append("🟢 Fresh tyre push lap (likely just pitted)")
    
    # Old tire incident
    if row['TyreLife'] >= 15 and row['DeviationSigma'] > 2:
        reasons.append("🔴 Severe tire degradation")
    
    # Big positive deviation
    if row['DeviationSigma'] > 2.5:
        reasons.append("🚨 Major slowdown — possible damage / dirty air / traffic")
    
    # Big negative deviation
    if row['DeviationSigma'] < -2.5:
        reasons.append("⚡ Stunning pace — qualifying-style push lap")
    
    return " | ".join(reasons) if reasons else "❓ Unexplained anomaly — requires review"

iso_anomalies['Reasoning'] = iso_anomalies.apply(explain_anomaly, axis=1)

print("🏁 INCIDENT REPORT — 2024 Bahrain GP\n")
print("=" * 100)

incident_report = iso_anomalies[[
    'Driver', 'LapNumber', 'LapTimeSeconds', 
    'TyreLife', 'Compound', 'DeviationSigma', 'Reasoning'
]].sort_values('LapNumber').reset_index(drop=True)

incident_report

🏁 INCIDENT REPORT — 2024 Bahrain GP



,Driver,LapNumber,LapTimeSeconds,TyreLife,Compound,DeviationSigma,Reasoning
0,VER,2.0,96.296,5.0,SOFT,0.601687,🟡 Race start lap (formation effects)
1,MAG,2.0,98.911,2.0,SOFT,1.359023,🟡 Race start lap (formation effects) | 🟢 Fresh...
2,ALB,2.0,98.826,2.0,SOFT,1.180073,🟡 Race start lap (formation effects) | 🟢 Fresh...
3,ZHO,2.0,98.920,2.0,SOFT,1.729143,🟡 Race start lap (formation effects) | 🟢 Fresh...
4,OCO,2.0,99.094,2.0,SOFT,1.372193,🟡 Race start lap (formation effects) | 🟢 Fresh...
5,STR,2.0,97.987,2.0,SOFT,0.931965,🟡 Race start lap (formation effects) | 🟢 Fresh...
6,TSU,2.0,98.689,2.0,SOFT,1.314754,🟡 Race start lap (formation effects) | 🟢 Fresh...
7,GAS,2.0,99.059,2.0,SOFT,1.293510,🟡 Race start lap (formation effects) | 🟢 Fresh...
8,BOT,2.0,98.259,2.0,SOFT,0.554415,🟡 Race start lap (formation effects) | 🟢 Fresh...
9,RIC,2.0,99.509,5.0,SOFT,1.650794,🟡 Race start lap (formation effects)


In [8]:
import joblib
import os

os.makedirs('../models', exist_ok=True)

# Save Isolation Forest
joblib.dump(iso_forest, '../models/incident_isolation_forest.pkl')
joblib.dump(scaler, '../models/incident_scaler.pkl')

# Save the incident report as CSV
incident_report.to_csv('../data/bahrain_2024_incidents.csv', index=False)

# Save high-confidence anomalies
both_flagged.to_csv('../data/high_confidence_incidents.csv', index=False)

print("✅ Incident detection system saved")
print(f"📁 Models: ../models/incident_isolation_forest.pkl")
print(f"📁 Report: ../data/bahrain_2024_incidents.csv ({len(incident_report)} incidents)")
print(f"📁 High-confidence: ../data/high_confidence_incidents.csv ({len(both_flagged)} incidents)")

# Summary stats
print("\n📊 DETECTION SUMMARY")
print(f"   Total laps analyzed: {len(all_laps_with_iso)}")
print(f"   Statistical anomalies: {all_laps_with_iso['IsAnomaly'].sum()}")
print(f"   ML anomalies: {len(iso_anomalies)}")
print(f"   Both methods agree: {len(both_flagged)}")
print(f"   Detection rate: {len(iso_anomalies)/len(all_laps_with_iso)*100:.1f}%")

✅ Incident detection system saved
📁 Models: ../models/incident_isolation_forest.pkl
📁 Report: ../data/bahrain_2024_incidents.csv (52 incidents)
📁 High-confidence: ../data/high_confidence_incidents.csv (8 incidents)

📊 DETECTION SUMMARY
   Total laps analyzed: 1024
   Statistical anomalies: 13
   ML anomalies: 52
   Both methods agree: 8
   Detection rate: 5.1%
